# 04 — MC Dropout Uncertainty, Uncertainty Validation and Selective Prediction (Phases 5–6)
**Project:** From Prediction to Referral — Uncertainty-Aware Pneumonia Detection under Cross-Population Distribution Shift

**Pre-specified before any uncertainty output existed (2026-10-06):**
| Item | Choice |
|---|---|
| MC Dropout | 50 passes; only the two head `nn.Dropout` layers active; backbone (BatchNorm/LayerNorm/stochastic depth) in eval mode |
| Classifier being audited | deterministic model at locked threshold A (identical to Notebook 03) |
| Uncertainty scores | **primary: predictive entropy of the MC mean (bits)**; secondary: mutual information (epistemic), MC variance; baseline: entropy of the deterministic probability |
| Hypotheses (tested, not assumed) | H1 uncertainty OOD > ID (OOD-detection AUROC, FPR@95TPR, ID test = 0 vs OOD = 1); H2 uncertainty error > correct (error-detection AUROC/AUPRC) |
| Referral thresholds | from **Mooney validation only**: τ_c = c-quantile of validation uncertainty, c ∈ {100, 95, 90, 80, 70, 60, 50} %; accept if u ≤ τ_c |
| Primary referral policy | τ at nominal 90 % coverage |
| Confidently incorrect | error with u ≤ τ_90 (an error the policy would not refer) → safety group G4, used by Notebook 05 |
| Risk–coverage | curves and AURC rank images within each cohort; E-AURC = AURC − optimal AURC |
| CIs | patient-level bootstrap, B = 2000 (OOD detection: independent ID and OOD draws) |

**Kaggle inputs:** the same five inputs as Notebook 03 (Notebook 02 must be the locked version; a guard checks this). The Notebook 03 output is NOT needed.
Also saves `predictions/image_logits_locked_version.csv`, from which the Notebook 03 statistics are recomputed for the locked model version.
GPU T4 ×1. Internet not needed.

In [ ]:
# ------------------------------------------------------------------ CONFIG
from pathlib import Path
INPUT_ROOT = Path("/kaggle/input")
NIH_ROOT   = Path("/kaggle/input/datasets/nih-chest-xrays/data")
PROJECT    = Path("/kaggle/working/CXR_OOD_Project")
MODELS     = ["efficientnet_b4", "vit_b16"]
N_BOOT     = 2000
BOOT_SEED  = 20261005
ECE_BINS   = 15
BATCH      = 64
SMOKE_TEST = False
T_MC       = 50                      # stochastic forward passes per image
SEED       = 42
COVERAGES  = [1.00, 0.95, 0.90, 0.80, 0.70, 0.60, 0.50]
MEASURES   = ["entropy_mc", "mi_mc", "var_mc", "entropy_det"]
PRIMARY    = "entropy_mc"            # pre-specified primary uncertainty score
POLICY     = 0.90                    # pre-specified referral policy: validation-derived threshold at nominal 90% coverage

LOCKED_THRESHOLDS = {
    "efficientnet_b4": {"A_primary": 0.4689939068127956, "B_secondary": 3.875000849286004, "C_default": 0.0},
    "vit_b16":         {"A_primary": -0.28979498040711366, "B_secondary": 3.5078123195824205, "C_default": 0.0},
}   # threshold_logit values
THR_A = {m: v["A_primary"] for m, v in LOCKED_THRESHOLDS.items()}
THR_LABEL = {"A_primary": "A (primary): max sens | spec≥0.95", "B_secondary": "B: max spec | sens≥0.95", "C_default": "C: 0.50"}
EXPECTED_FP = {
    "MOONEY_VAL": "13d0556c2de25782967c5cd20ceb54508ca4b8061cdca9e0c9c6ae0bff2f139c",
    "MOONEY_TEST": "723c6c2e962c6704b4e61e693ea96f831104cd9f937acbbf309b46b0edc55101",
    "OOD_AP_ALLAGE": "2f209001f1183bcddc265cd93a36eaf00d2151d6d0088b0b44c2cd1d03c70f80",
    "OOD_AP_ADULT": "b13bc8c6413b3a5725e08184ade4a34305288d0daff717533589399c83e0d567",
    "OOD_PA_ALLAGE": "f1a6c96829c1b91d1221cd971c2de053be878f49fea846ff5fd36ba3e36b8474",
    "OOD_PA_ADULT": "ede8eeb671334525f5d4d0edce8a8a025bcf7992bb19d9060e8a17c6bd116cb0",
    "OOD_HARD_AP_ALLAGE": "9647dd49e598ef43d1321fe31e4401786c8293e15ba71956d8b6c77f2c922372",
    "OOD_HARD_AP_ADULT": "b8cc02779f56de273cd8ffcca8bc5915451e279f97f9b42f2211c1f1c544fb60",
}
ID_COHORTS  = ["MOONEY_TEST", "MOONEY_TEST_NONCOLLIDING", "MOONEY_TEST_DEDUP"]
OOD_PRIMARY = ["OOD_AP_ADULT", "OOD_PA_ADULT", "OOD_HARD_AP_ADULT"]
OOD_SENS    = ["OOD_AP_ALLAGE", "OOD_PA_ALLAGE", "OOD_HARD_AP_ALLAGE"]
EVAL_COHORTS = ID_COHORTS + OOD_PRIMARY + OOD_SENS

In [ ]:
import os, re, sys, json, time, math, hashlib, shutil, platform, datetime
import numpy as np, pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from scipy.optimize import minimize_scalar
from scipy.stats import norm
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

DIRS = {k: PROJECT / k for k in ["predictions", "metrics", "figures", "tables", "logs", "configs", "code"]}
for d in DIRS.values(): d.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda" and not SMOKE_TEST:
    raise SystemExit("No GPU detected — set Accelerator to GPU T4 and restart.")

PRUNE = re.compile(r"^(images(_\d+)?|train|test|val|NORMAL|PNEUMONIA|chest_xray|__MACOSX)$")
def find_one(name):
    hits = []
    for root, dirs, files in os.walk(INPUT_ROOT):
        dirs[:] = [d for d in dirs if not PRUNE.match(d)]
        if name in files: hits.append(Path(root) / name)
    if not hits:
        raise SystemExit(f"'{name}' not found under {INPUT_ROOT}. Check Add Input.")
    digests = {hashlib.sha256(h.read_bytes()).hexdigest() for h in hits}
    if len(digests) > 1:
        raise SystemExit(f"Found {len(hits)} DIFFERENT copies of '{name}': {hits}. Remove the wrong input.")
    if len(hits) > 1: print(f"note: {len(hits)} byte-identical copies of {name}; using {sorted(hits)[0]}")
    return sorted(hits)[0]

FILES = {n: find_one(n) for n in ["mooney_manifest_final.csv", "cxr_common.py"] +
         [f"{m}_best.pt" for m in MODELS] + [f"{m}_val_predictions.csv" for m in MODELS] +
         [f"{c}.csv" for c in OOD_PRIMARY + OOD_SENS]}
for k, v in FILES.items(): print(f"{k:32s} {v}")

# ---- guard: Notebook 02 input must be the version whose validation predictions were used to lock the thresholds (2026-10-05)
EXPECTED_VAL_PRED_SHA = {
    "efficientnet_b4": "33a07785b024d7408591941c4461bd314a9ff63988628709b90efcb7e12ea5fd",
    "vit_b16": "cc21299bc2cd00b7555eec65ab34f61df2488d59cd78ebaba678e8d8cc58c43f"
}
for _m, _h in EXPECTED_VAL_PRED_SHA.items():
    _got = hashlib.sha256(FILES[f"{_m}_val_predictions.csv"].read_bytes()).hexdigest()
    print(f"[{'PASS' if _got == _h else 'FAIL'}] Notebook 02 version check ({_m})")
    if _got != _h and not SMOKE_TEST:
        raise SystemExit(f"Wrong Notebook 02 version attached: {_m}_val_predictions.csv differs from the version used to lock the "
                         "thresholds. In 'Input', switch the Notebook 02 output to the version whose ZIP was reviewed (logs/environment_02.json "
                         "timestamp 2026-10-05T08:09:40) and rerun.")
shutil.copy(FILES["cxr_common.py"], DIRS["code"] / "cxr_common.py")
sys.path.insert(0, str(DIRS["code"]))
import importlib, cxr_common; importlib.reload(cxr_common)
from cxr_common import load_letterboxed, to_model_tensor, CXRClassifier

ENV = dict(python=platform.python_version(), torch=torch.__version__, numpy=np.__version__, pandas=pd.__version__,
           gpu=torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu",
           timestamp_utc=datetime.datetime.now(datetime.timezone.utc).isoformat())
(DIRS["logs"] / "environment_04.json").write_text(json.dumps(ENV, indent=2))

## 1. Build and verify cohorts

In [ ]:
def fingerprint(paths): return hashlib.sha256("\n".join(sorted(paths)).encode()).hexdigest()

man = pd.read_csv(FILES["mooney_manifest_final.csv"])
dev_pids = set(man.loc[man.final_split.isin(["train", "val"]), "patient_id"].astype(str))
COH = {}
for name, split in [("MOONEY_VAL", "val"), ("MOONEY_TEST", "test_id")]:
    d = man[man.final_split == split].sort_values("filepath").reset_index(drop=True)
    COH[name] = pd.DataFrame({"dataset": "MOONEY_KERMANY", "cohort": name, "filename": d.filename, "filepath": d.filepath,
                              "patient_id": "M:" + d.patient_id.astype(str), "y": d.y.astype(int), "sha256": d.sha256})
t = COH["MOONEY_TEST"]
COH["MOONEY_TEST_NONCOLLIDING"] = t[~t.patient_id.str[2:].isin(dev_pids)].assign(cohort="MOONEY_TEST_NONCOLLIDING")
COH["MOONEY_TEST_DEDUP"] = t[~t.sha256.duplicated(keep="first")].assign(cohort="MOONEY_TEST_DEDUP")
del man

nih_index = {p.name: str(p) for p in NIH_ROOT.glob("images_*/images/*.png")}
print("NIH files indexed:", len(nih_index))
for c in OOD_PRIMARY + OOD_SENS:
    d = pd.read_csv(FILES[f"{c}.csv"]).sort_values("filename").reset_index(drop=True)
    missing = (~d.filename.isin(nih_index)).sum()
    if missing: raise SystemExit(f"{c}: {missing} images not found under NIH_ROOT")
    COH[c] = pd.DataFrame({"dataset": "NIH_CHESTXRAY14", "cohort": c, "filename": d.filename,
                           "filepath": d.filename.map(nih_index), "patient_id": "N:" + d.patient_id.astype(str),
                           "y": d.y.astype(int), "age": d.age, "sex": d.sex, "analysis_role": d.analysis_role})

checks = []
for c in ["MOONEY_VAL", "MOONEY_TEST"]:
    checks.append((c, fingerprint(COH[c].filepath) == EXPECTED_FP[c]))
for c in OOD_PRIMARY + OOD_SENS:
    checks.append((c, hashlib.sha256("\n".join(COH[c].filename).encode()).hexdigest() == EXPECTED_FP[c]))
checks += [("MOONEY_TEST_NONCOLLIDING n=304", len(COH["MOONEY_TEST_NONCOLLIDING"]) == 304),
           ("MOONEY_TEST_DEDUP n=618", len(COH["MOONEY_TEST_DEDUP"]) == 618)]
for c, ok in checks: print(f"[{'PASS' if ok else 'FAIL'}] cohort {c}")
if not all(ok for _, ok in checks) and not SMOKE_TEST:
    raise SystemExit("Cohort verification failed — inputs differ from the locked Phase 1 cohorts.")
if SMOKE_TEST:
    COH = {k: (v.groupby("y", group_keys=False).head(15) if len(v) > 30 else v).reset_index(drop=True) for k, v in COH.items()}

## 2. Inference — every unique image decoded once, both models in the same pass

In [ ]:
SIZES = {m: torch.load(FILES[f"{m}_best.pt"], map_location="cpu", weights_only=False)["config"]["img_size"] for m in MODELS}

def load_multi(path, sizes):
    """Identical operations to cxr_common.load_letterboxed, decoding once for several sizes."""
    im = Image.open(path).convert("L")
    w, h = im.size; s = max(w, h)
    canvas = Image.new("L", (s, s), 0); canvas.paste(im, ((s - w) // 2, (s - h) // 2))
    return {z: np.asarray(canvas.resize((z, z), Image.LANCZOS), dtype=np.uint8) for z in sizes}

_probe = FILES["mooney_manifest_final.csv"]  # equivalence check on real images
for p in list(COH["MOONEY_VAL"].filepath[:2]) + list(COH["OOD_AP_ADULT"].filepath[:2]):
    mm = load_multi(p, set(SIZES.values()))
    for z in SIZES.values(): assert np.array_equal(mm[z], load_letterboxed(p, z)), "preprocessing mismatch"
print("preprocessing identical to Notebook 02:", True)

class Imgs(Dataset):
    def __init__(self, paths, sizes): self.paths, self.sizes = list(paths), sorted(set(sizes))
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        a = load_multi(self.paths[i], self.sizes)
        return {str(z): torch.from_numpy(a[z]).unsqueeze(0) for z in self.sizes}, i

NETS = {}
for m in MODELS:
    st = torch.load(FILES[f"{m}_best.pt"], map_location="cpu", weights_only=False); cfg = st["config"]
    net = CXRClassifier(cfg["timm_name"], pretrained=False, drop_path=cfg["drop_path"],
                        head_dropout=cfg["head_dropout"], head_hidden=cfg["head_hidden"])
    net.load_state_dict(st["model_state"]); NETS[m] = net.to(DEVICE).eval()
    print(f"{m}: epoch {st['epoch']}, val loss {st['val_loss']:.4f}, img {cfg['img_size']}")

## 1b. Numeric-mode check: reproduce Notebook 02 validation logits before any inference
The Kaggle environment can change between sessions. For each model, the first computation mode that reproduces the
training-time validation logits (Notebook 02) is selected and used for all inference below.

In [ ]:
import contextlib, PIL, timm
from torch.nn.attention import sdpa_kernel, SDPBackend
print("torch", torch.__version__, "| timm", timm.__version__, "| pillow", PIL.__version__, "| cuda", torch.version.cuda)
def mode_ctx(mode):
    st = contextlib.ExitStack()
    if "math" in mode: st.enter_context(sdpa_kernel(SDPBackend.MATH))
    if mode.startswith("fp16"): st.enter_context(torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"))
    return st
MODES = ["fp16_autocast", "fp32", "fp16_sdpa_math", "fp32_sdpa_math"]
MODE, MODE_REPORT = {}, {}
probe = COH["MOONEY_VAL"].head(64)
for m in MODELS:
    ref02 = pd.read_csv(FILES[f"{m}_val_predictions.csv"]).set_index("filepath")["logit"].loc[probe.filepath].values
    x = to_model_tensor(torch.stack([torch.from_numpy(load_letterboxed(p, SIZES[m])).unsqueeze(0) for p in probe.filepath]).to(DEVICE))
    MODE_REPORT[m] = {}
    for md in MODES:
        with torch.no_grad(), mode_ctx(md):
            z = NETS[m](x).float().cpu().numpy()
        d = np.abs(z - ref02); MODE_REPORT[m][md] = dict(median=float(np.median(d)), max=float(d.max()))
        print(f"{m:16s} {md:15s} vs Notebook 02 validation logits: median |Δ| {np.median(d):.4f}, max {d.max():.4f}")
    ok = [md for md in MODES if MODE_REPORT[m][md]["median"] < 0.01 and MODE_REPORT[m][md]["max"] < 0.1]
    if not ok and not SMOKE_TEST:
        raise SystemExit(f"{m}: no computation mode reproduces the Notebook 02 logits in this Kaggle environment "
                         f"(torch {torch.__version__}, timm {timm.__version__}). Pin this notebook's environment to the one used "
                         "by Notebook 02/03 (Settings → Environment) and rerun.")
    MODE[m] = ok[0] if ok else MODES[0]
    print(f"→ {m}: using {MODE[m]}")
(DIRS["logs"] / "numeric_mode_04.json").write_text(json.dumps(dict(selected=MODE, report=MODE_REPORT,
    torch=torch.__version__, timm=timm.__version__, pillow=PIL.__version__), indent=2))

## 2. MC Dropout inference
Backbone features are computed once per image in eval mode (BatchNorm, LayerNorm and stochastic depth deterministic).
Only the two head `nn.Dropout` layers are switched to train mode, and the head is sampled `T_MC` times.
The deterministic head (dropout off) is also computed and must reproduce Notebook 03's logits.

In [ ]:
def mc_head(net, f, T, gen_seed):
    """f: (B,F) float32 features. Returns deterministic logits (B,) and MC logits (T,B)."""
    net.eval()
    det = net.head(f)
    for d in net.mc_dropout_modules(): d.train()
    torch.manual_seed(gen_seed)
    mc = torch.stack([net.head(f) for _ in range(T)])
    for d in net.mc_dropout_modules(): d.eval()
    return det, mc

@torch.no_grad()
def run_mc(paths, chunk_id):
    dl = DataLoader(Imgs(paths, SIZES.values()), batch_size=BATCH, num_workers=min(4, os.cpu_count()), pin_memory=True)
    det = {m: np.empty(len(paths), np.float32) for m in MODELS}
    mc = {m: np.empty((len(paths), T_MC), np.float32) for m in MODELS}
    t0, n = time.time(), 0
    for b, (xs, idx) in enumerate(dl):
        for j, (m, net) in enumerate(NETS.items()):
            x = to_model_tensor(xs[str(SIZES[m])].to(DEVICE, non_blocking=True))
            with mode_ctx(MODE[m]):
                f = net.features(x)
                dz = net.head(f).float()            # deterministic logit in the validated numeric mode
            _, mz = mc_head(net, f.float(), T_MC, SEED * 1_000_003 + chunk_id * 10_007 + b * 101 + j)
            det[m][idx.numpy()] = dz.cpu().numpy(); mc[m][idx.numpy()] = mz.T.cpu().numpy()
        n += len(idx)
        if b % 100 == 0: print(f"  chunk {chunk_id}: {n}/{len(paths)} images, {n / (time.time() - t0):.0f} img/s")
    return det, mc

MC_DIR = PROJECT / "mc_cache"; MC_DIR.mkdir(exist_ok=True)
CACHE_VER = "v5-stable-entropy"
all_paths = sorted(set(pd.concat([c.filepath for c in COH.values()])))
CHUNK = 10000
for ci_, i in enumerate(range(0, len(all_paths), CHUNK)):
    f = MC_DIR / f"chunk_{ci_:03d}.npz"
    part = all_paths[i:i + CHUNK]
    if f.exists():
        z = np.load(f, allow_pickle=True)
        if "ver" in z and str(z["ver"]) == CACHE_VER and list(z["paths"]) == part: continue
    det, mc = run_mc(part, ci_)
    np.savez(f, ver=np.array(CACHE_VER), paths=np.array(part, dtype=object), **{f"det_{m}": det[m] for m in MODELS},
             **{f"mc_{m}": mc[m].astype(np.float16) for m in MODELS})
    print(f"saved {f.name}")

DET, MCS, PATHS = {m: [] for m in MODELS}, {m: [] for m in MODELS}, []
for f in sorted(MC_DIR.glob("chunk_*.npz")):
    z = np.load(f, allow_pickle=True); PATHS += list(z["paths"])
    for m in MODELS: DET[m].append(z[f"det_{m}"]); MCS[m].append(z[f"mc_{m}"].astype(np.float32))
assert PATHS == all_paths, "MC cache does not match the current image list — delete mc_cache/ and rerun"
DET = {m: np.concatenate(v) for m, v in DET.items()}; MCS = {m: np.concatenate(v) for m, v in MCS.items()}
POS = {p: i for i, p in enumerate(PATHS)}

# Deterministic logits of the locked Notebook 02 version (validated above against its own validation logits).
# They supersede the Notebook 03 logits, which were produced with a different Notebook 02 run (identified 2026-10-21).
pd.DataFrame({"filepath": PATHS, **{m: DET[m] for m in MODELS}}).to_csv(DIRS["predictions"] / "image_logits_locked_version.csv", index=False)
REPRO = {m: "validated against Notebook 02 validation logits (logs/numeric_mode_04.json)" for m in MODELS}

## 3. Per-image uncertainty
Entropies in bits (range 0–1). `entropy_mc` = H(mean_t p_t) [primary]; `mi_mc` = H(p̄) − mean_t H(p_t) (epistemic);
`var_mc` = Var_t(p_t); `entropy_det` = H(p_deterministic) (baseline without MC sampling).
Prediction and correctness use the deterministic logit at the locked threshold A, exactly as in Notebook 03.

In [ ]:
from scipy.special import xlogy, expit
def H_pq(p, q):
    """Binary entropy in bits from p and q = 1 - p, both computed separately so neither underflows to an exact 0/1 error."""
    return -(xlogy(p, p) + xlogy(q, q)) / np.log(2)
UNC = {}
for m in MODELS:
    z = MCS[m].astype(np.float64); p, q = expit(z), expit(-z)          # float64, numerically stable
    pbar, qbar = p.mean(1), q.mean(1)
    zd = DET[m].astype(np.float64)
    UNC[m] = pd.DataFrame({"filepath": PATHS, "logit_det": DET[m], "prob_det": expit(zd), "prob_mc_mean": pbar,
                           "var_mc": p.var(1), "entropy_mc": H_pq(pbar, qbar),
                           "mi_mc": np.clip(H_pq(pbar, qbar) - H_pq(p, q).mean(1), 0, None),
                           "entropy_det": H_pq(expit(zd), expit(-zd))})
    bad = UNC[m][["var_mc", "entropy_mc", "mi_mc", "entropy_det"]].isna().sum()
    if bad.any(): raise SystemExit(f"{m}: NaN in uncertainty measures {bad[bad > 0].to_dict()} — stop.")
np.savez_compressed(DIRS["predictions"] / "mc_logits_float16.npz", paths=np.array(PATHS, dtype=object),
                    **{m: MCS[m].astype(np.float16) for m in MODELS}, **{f"det_{m}": DET[m] for m in MODELS})
rows = []
for m in MODELS:
    u = UNC[m].set_index("filepath")
    for c in ["MOONEY_VAL"] + EVAL_COHORTS:
        d = COH[c][["dataset", "cohort", "filename", "filepath", "patient_id", "y"]].copy(); d.insert(0, "model", m)
        d = d.join(u, on="filepath")
        d["pred"] = (d.logit_det >= THR_A[m]).astype(int)
        d["error"] = (d.pred != d.y).astype(int)
        d["error_type"] = np.where(d.error == 0, "correct", np.where(d.y == 1, "FN", "FP"))
        rows.append(d)
U = pd.concat(rows, ignore_index=True)

# referral thresholds from Mooney validation only: accept if u <= tau_c, tau_c = c-quantile of validation uncertainty
TAU = {}
for m in MODELS:
    v = U[(U.model == m) & (U.cohort == "MOONEY_VAL")]
    TAU[m] = {meas: {f"{int(c * 100)}": float(np.quantile(v[meas], c, method="higher")) for c in COVERAGES} for meas in MEASURES}
(DIRS["configs"] / "uncertainty_thresholds_from_validation.json").write_text(json.dumps(dict(
    thresholds=TAU, rule="accept if uncertainty <= tau_c; tau_c = c-quantile (method='higher') of Mooney validation uncertainty",
    primary_measure=PRIMARY, policy_coverage=POLICY, T_MC=T_MC, seed=SEED), indent=2))
pol = {m: TAU[m][PRIMARY][str(int(POLICY * 100))] for m in MODELS}
U["low_uncertainty"] = [int(u <= pol[m]) for m, u in zip(U.model, U[PRIMARY])]
U["safety_group"] = np.select(
    [(U.error == 0) & (U.low_uncertainty == 1), (U.error == 0) & (U.low_uncertainty == 0),
     (U.error == 1) & (U.low_uncertainty == 0), (U.error == 1) & (U.low_uncertainty == 1)],
    ["G1_correct_low_unc", "G2_correct_high_unc", "G3_incorrect_high_unc", "G4_incorrect_low_unc"], default="NA")
U.to_csv(DIRS["predictions"] / "uncertainty_per_image.csv.gz", index=False)
print("Referral threshold (primary measure, nominal 90% coverage):", pol)
print(U[U.cohort.isin(["MOONEY_TEST"] + OOD_PRIMARY)].groupby(["model", "cohort"]).safety_group.value_counts().unstack().fillna(0).astype(int))

## 4. Statistics with patient-level bootstrap
Error detection (target = incorrect, score = uncertainty) per cohort; OOD detection (MOONEY_TEST = 0 vs OOD cohort = 1);
risk–coverage (AURC, ranking within cohort); and the operational referral policy (thresholds transported from validation).

In [ ]:
def bin_scores(s):
    u, inv = np.unique(s, return_inverse=True); return inv, len(u)

def w_auc_ap(inv, K, lab, w):
    pos = np.bincount(inv, w * lab, K); neg = np.bincount(inv, w * (1 - lab), K)
    P, N = pos.sum(), neg.sum()
    if P <= 0 or N <= 0: return np.nan, np.nan, np.nan
    auc = float(np.sum(pos * (np.cumsum(neg) - neg + 0.5 * neg)) / (P * N))
    tp, fp = np.cumsum(pos[::-1]), np.cumsum(neg[::-1])
    prec = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
    ap = float(np.sum(np.diff(np.r_[0, tp / P]) * prec))
    k = np.searchsorted(tp / P, 0.95 - 1e-12)              # first threshold (descending score) reaching TPR >= 0.95
    fpr95 = float(fp[min(k, len(fp) - 1)] / N)
    return auc, ap, fpr95

def w_aurc(inv, K, err, w):
    """Ranking ascending by uncertainty; ties accepted together."""
    e = np.bincount(inv, w * err, K); n = np.bincount(inv, w, K)
    ce, cn = np.cumsum(e), np.cumsum(n)
    risk = np.divide(ce, cn, out=np.zeros_like(ce), where=cn > 0)
    return float(np.sum(n * risk) / cn[-1])

def aurc_optimal(n, k):
    if k == 0: return 0.0
    i = np.arange(n - k + 1, n + 1); return float(np.sum((i - (n - k)) / i) / n)

def selective(d, tau, w):
    acc = (d["u"].values <= tau); y, yh, err = d.y.values, d.pred.values, d.error.values
    W = w.sum(); Wa = (w * acc).sum(); E = (w * err).sum()
    sd = lambda a, b: a / b if b > 0 else np.nan
    return dict(coverage=sd(Wa, W), selective_risk=sd((w * acc * err).sum(), Wa),
                sens_accepted=sd((w * acc * (y == 1) * (yh == 1)).sum(), (w * acc * (y == 1)).sum()),
                spec_accepted=sd((w * acc * (y == 0) * (yh == 0)).sum(), (w * acc * (y == 0)).sum()),
                errors_referred_pct=sd((w * (~acc) * err).sum(), E) * 100 if E > 0 else np.nan,
                confidently_incorrect_pct_of_all=sd((w * acc * err).sum(), W) * 100)

def cohort_seed(c): return BOOT_SEED + int(hashlib.md5(c.encode()).hexdigest()[:8], 16) % 10**6
def draws_for(c, nb):
    rng = np.random.default_rng(cohort_seed(c)); npat = COH[c].patient_id.nunique()
    codes = pd.factorize(COH[c].patient_id.values)[0]
    return codes, [np.bincount(rng.integers(0, npat, npat), minlength=npat).astype(np.int32) for _ in range(nb)]
NB = N_BOOT if not SMOKE_TEST else 50
DRAWS = {c: draws_for(c, NB) for c in EVAL_COHORTS}

def ci(a):
    a = np.asarray(a, float); a = a[~np.isnan(a)]
    return (np.percentile(a, 2.5), np.percentile(a, 97.5)) if len(a) else (np.nan, np.nan)
def fmt(v, lo, hi, d=3): return f"{v:.{d}f} ({lo:.{d}f}–{hi:.{d}f})" if not (v is None or np.isnan(v)) else "NA"

res, t0 = [], time.time()
for m in MODELS:
    for c in EVAL_COHORTS:
        d = U[(U.model == m) & (U.cohort == c)].reset_index(drop=True)
        codes, dr = DRAWS[c]
        assert len(d) == len(codes)
        for meas in MEASURES:
            inv, K = bin_scores(d[meas].values); err = d.error.values.astype(float)
            pt_auc, pt_ap, _ = w_auc_ap(inv, K, err, np.ones(len(d))); pt_aurc = w_aurc(inv, K, err, np.ones(len(d)))
            bs = np.array([(lambda w: (*w_auc_ap(inv, K, err, w)[:2], w_aurc(inv, K, err, w)))(cnt[codes].astype(float)) for cnt in dr])
            for k, v, col in [("errdet_auroc", pt_auc, 0), ("errdet_auprc", pt_ap, 1), ("aurc", pt_aurc, 2)]:
                lo, hi = ci(bs[:, col]); res.append(dict(model=m, cohort=c, measure=meas, metric=k, estimate=v, ci_low=lo, ci_high=hi))
            res.append(dict(model=m, cohort=c, measure=meas, metric="aurc_optimal", estimate=aurc_optimal(len(d), int(err.sum())), ci_low=np.nan, ci_high=np.nan))
            res.append(dict(model=m, cohort=c, measure=meas, metric="error_rate", estimate=float(err.mean()), ci_low=np.nan, ci_high=np.nan))
        if c != "MOONEY_TEST" and c not in ID_COHORTS:
            dI = U[(U.model == m) & (U.cohort == "MOONEY_TEST")].reset_index(drop=True); cI, drI = DRAWS["MOONEY_TEST"]
            lab = np.r_[np.zeros(len(dI)), np.ones(len(d))]
            for meas in MEASURES:
                inv, K = bin_scores(np.r_[dI[meas].values, d[meas].values])
                pt = w_auc_ap(inv, K, lab, np.ones(len(lab)))
                bs = np.array([w_auc_ap(inv, K, lab, np.r_[a[cI], b[codes]].astype(float)) for a, b in zip(drI, dr)])
                for k, col in [("ooddet_auroc", 0), ("ooddet_fpr95", 2)]:
                    lo, hi = ci(bs[:, col]); res.append(dict(model=m, cohort=c, measure=meas, metric=k, estimate=pt[col], ci_low=lo, ci_high=hi))
        print(f"{m} {c}: done ({time.time() - t0:.0f}s)")
STAT = pd.DataFrame(res); STAT.to_csv(DIRS["metrics"] / "uncertainty_metrics_with_ci_long.csv", index=False)

sel = []
for m in MODELS:
    for c in ["MOONEY_VAL"] + EVAL_COHORTS:
        d = U[(U.model == m) & (U.cohort == c)].reset_index(drop=True)
        for meas in [PRIMARY, "entropy_det"]:
            dd = d.assign(u=d[meas])
            for cov in COVERAGES:
                tau = TAU[m][meas][str(int(cov * 100))]
                pt = selective(dd, tau, np.ones(len(dd)))
                row = dict(model=m, cohort=c, measure=meas, nominal_coverage_pct=int(cov * 100), tau=tau,
                           n_total=len(dd), n_retained=int((dd.u <= tau).sum()), n_referred=int((dd.u > tau).sum()),
                           n_errors=int(dd.error.sum()), n_errors_referred=int(((dd.u > tau) & (dd.error == 1)).sum()), **pt)
                if c in DRAWS and meas == PRIMARY:
                    codes, dr = DRAWS[c]
                    bs = pd.DataFrame([selective(dd, tau, cnt[codes].astype(float)) for cnt in dr])
                    for k in ["coverage", "selective_risk", "sens_accepted", "spec_accepted", "errors_referred_pct"]:
                        row[f"{k}_ci"] = "–".join(f"{x:.3f}" for x in ci(bs[k]))
                sel.append(row)
SEL = pd.DataFrame(sel); SEL.to_csv(DIRS["metrics"] / "selective_prediction_validation_thresholds.csv", index=False)

## 5. Hypothesis tests reported descriptively + tables 5–7

In [ ]:
def get(m, c, meas, k):
    r = STAT[(STAT.model == m) & (STAT.cohort == c) & (STAT.measure == meas) & (STAT.metric == k)]
    return r.iloc[0] if len(r) else None
MAIN = ["MOONEY_TEST"] + OOD_PRIMARY

t5 = []
for m in MODELS:
    for c in MAIN:
        d = U[(U.model == m) & (U.cohort == c)]
        row = dict(model=m, cohort=c, n=len(d), errors=int(d.error.sum()), error_rate=round(d.error.mean(), 4))
        for meas in MEASURES:
            q = d[meas].quantile([.25, .5, .75]).values
            row[f"{meas} median (IQR)"] = f"{q[1]:.4f} ({q[0]:.4f}–{q[2]:.4f})"
        for grp in ["correct", "error"]:
            dd = d[d.error == (grp == "error")]
            row[f"{PRIMARY} median, {grp}"] = round(float(dd[PRIMARY].median()), 4) if len(dd) else np.nan
        t5.append(row)
pd.DataFrame(t5).to_csv(DIRS["tables"] / "table5_uncertainty_characteristics.csv", index=False)

t6 = []
for m in MODELS:
    for c in MAIN:
        for meas in MEASURES:
            row = dict(model=m, cohort=c, measure=meas)
            for k, lab in [("errdet_auroc", "Error-detection AUROC"), ("errdet_auprc", "Error-detection AUPRC"),
                           ("ooddet_auroc", "OOD-detection AUROC (vs ID test)"), ("ooddet_fpr95", "OOD FPR@95%TPR")]:
                r = get(m, c, meas, k); row[lab] = fmt(r.estimate, r.ci_low, r.ci_high) if r is not None else "—"
            r = get(m, c, meas, "error_rate"); row["Error rate (chance AUPRC)"] = round(r.estimate, 4)
            t6.append(row)
pd.DataFrame(t6).to_csv(DIRS["tables"] / "table6_error_and_ood_detection.csv", index=False)

t7 = SEL[(SEL.measure == PRIMARY) & SEL.cohort.isin(MAIN)].copy()
for m in MODELS:
    for c in MAIN:
        r, o = get(m, c, PRIMARY, "aurc"), get(m, c, PRIMARY, "aurc_optimal")
        t7.loc[(t7.model == m) & (t7.cohort == c), "AURC (95% CI)"] = fmt(r.estimate, r.ci_low, r.ci_high, 4)
        t7.loc[(t7.model == m) & (t7.cohort == c), "E-AURC"] = round(r.estimate - o.estimate, 4)
t7.to_csv(DIRS["tables"] / "table7_selective_prediction.csv", index=False)

ci_rows = []
for m in MODELS:
    for c in MAIN:
        d = U[(U.model == m) & (U.cohort == c)]; e = d[d.error == 1]
        ci_rows.append(dict(model=m, cohort=c, errors=len(e), confidently_incorrect=int(e.low_uncertainty.sum()),
                            pct_of_errors=round(100 * e.low_uncertainty.mean(), 1) if len(e) else np.nan,
                            confident_FP=int(((e.error_type == "FP") & (e.low_uncertainty == 1)).sum()),
                            confident_FN=int(((e.error_type == "FN") & (e.low_uncertainty == 1)).sum()),
                            pct_of_all_images=round(100 * e.low_uncertainty.sum() / len(d), 2)))
pd.DataFrame(ci_rows).to_csv(DIRS["tables"] / "tableS_confidently_incorrect.csv", index=False)
print(pd.DataFrame(ci_rows).to_string(index=False))
print(pd.read_csv(DIRS["tables"] / "table6_error_and_ood_detection.csv").query("measure == @PRIMARY").to_string(index=False))

## 6. Figures 4–6

In [ ]:
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "savefig.dpi": 600, "figure.dpi": 120})
NICE = {"efficientnet_b4": "EfficientNet-B4", "vit_b16": "ViT-B/16"}
SHORT = {"MOONEY_TEST": "ID test", "OOD_AP_ADULT": "NIH AP", "OOD_PA_ADULT": "NIH PA", "OOD_HARD_AP_ADULT": "NIH AP-Hard"}
CC = {"MOONEY_TEST": "#222222", "OOD_AP_ADULT": "#1f5aa6", "OOD_PA_ADULT": "#2e8b57", "OOD_HARD_AP_ADULT": "#c0562b"}
def save(fig, name):
    for ext in ["png", "pdf"]: fig.savefig(DIRS["figures"] / f"{name}.{ext}", bbox_inches="tight")
    plt.close(fig)
def ecdf(ax, x, **kw):
    x = np.sort(x); ax.step(x, np.arange(1, len(x) + 1) / len(x), where="post", **kw)

fig, axes = plt.subplots(1, len(MODELS), figsize=(3.6 * len(MODELS), 3.0), squeeze=False)
for i, m in enumerate(MODELS):
    ax = axes[0, i]
    for c in MAIN: ecdf(ax, U[(U.model == m) & (U.cohort == c)][PRIMARY].values, color=CC[c], lw=1.3, label=SHORT[c])
    ax.axvline(pol[m], color="grey", ls="--", lw=0.8, label="Referral threshold (val, 90%)")
    ax.set(xlabel="MC-Dropout predictive entropy (bits)", ylabel="Cumulative fraction", title=NICE[m], xlim=(0, 1))
    ax.legend(frameon=False, fontsize=7, loc="lower right")
fig.tight_layout(); save(fig, "figure4_uncertainty_id_vs_ood")

fig, axes = plt.subplots(len(MODELS), len(MAIN), figsize=(2.4 * len(MAIN), 2.4 * len(MODELS)), squeeze=False, sharex=True)
for i, m in enumerate(MODELS):
    for j, c in enumerate(MAIN):
        ax = axes[i, j]; d = U[(U.model == m) & (U.cohort == c)]
        for e, col, lab in [(0, "#2e8b57", "Correct"), (1, "#b22222", "Incorrect")]:
            x = d[d.error == e][PRIMARY].values
            if len(x): ecdf(ax, x, color=col, lw=1.3, label=f"{lab} (n={len(x)})")
        ax.axvline(pol[m], color="grey", ls="--", lw=0.8)
        r = get(m, c, PRIMARY, "errdet_auroc"); ax.set_title(f"{SHORT[c]}\nerror-det. AUROC {r.estimate:.2f}", fontsize=8)
        ax.legend(frameon=False, fontsize=6.5, loc="lower right")
        if j == 0: ax.set_ylabel(f"{NICE[m]}\nCumulative fraction")
        if i == len(MODELS) - 1: ax.set_xlabel("Predictive entropy (bits)")
fig.tight_layout(); save(fig, "figure5_uncertainty_correct_vs_incorrect")

fig, axes = plt.subplots(1, len(MODELS), figsize=(3.6 * len(MODELS), 3.0), squeeze=False)
for i, m in enumerate(MODELS):
    ax = axes[0, i]
    for c in MAIN:
        d = U[(U.model == m) & (U.cohort == c)].sort_values(PRIMARY, kind="mergesort")
        n = np.arange(1, len(d) + 1); ax.plot(n / len(d), np.cumsum(d.error.values) / n, color=CC[c], lw=1.3,
                                              label=f"{SHORT[c]} (AURC {get(m, c, PRIMARY, 'aurc').estimate:.3f})")
        s = SEL[(SEL.model == m) & (SEL.cohort == c) & (SEL.measure == PRIMARY) & (SEL.nominal_coverage_pct == int(POLICY * 100))].iloc[0]
        ax.plot(s.coverage, s.selective_risk, "o", color=CC[c], ms=5, mec="white")
    ax.set(xlabel="Coverage (fraction accepted by AI)", ylabel="Selective risk (error rate)", title=NICE[m], xlim=(0, 1), ylim=(0, 1))
    ax.legend(frameon=False, fontsize=7, loc="best")
fig.text(0.5, -0.02, "Curves rank images within each cohort; dots = operating point of the validation-derived 90% referral threshold.",
         ha="center", fontsize=7)
fig.tight_layout(); save(fig, "figure6_risk_coverage")

## 7. Run log and ZIP

In [ ]:
summary = dict(T_MC=T_MC, seed=SEED, primary_measure=PRIMARY, policy_coverage=POLICY, referral_threshold=pol,
               thresholds_A_logit=THR_A, n_boot=NB, env=ENV, cohort_checks=dict(checks), reproduction=REPRO, numeric_mode=MODE)
(DIRS["metrics"] / "notebook04_summary.json").write_text(json.dumps(summary, indent=2, default=str))
zip_base = PROJECT / "phase5_6_uncertainty_outputs"
tmp = PROJECT / "_zip04"; shutil.rmtree(tmp, ignore_errors=True)
for k in ["predictions", "metrics", "figures", "tables", "logs", "configs"]:
    shutil.copytree(DIRS[k], tmp / k, dirs_exist_ok=True)
shutil.make_archive(str(zip_base), "zip", tmp); shutil.rmtree(tmp)
print("ZIP →", f"{zip_base}.zip", f"({os.path.getsize(f'{zip_base}.zip') / 1e6:.0f} MB)")
print("Phases 5–6 complete. Review before Notebook 05 (Grad-CAM).")